In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load datasets
train_df = pd.read_csv('train_(2)_(1)_(1).csv')
dist_df = pd.read_csv('dist_from_city_centre_(1)_(1)_(1).csv')
rent_df = pd.read_csv('avg_rent_(1)_(1)_(1).csv')
test_df = pd.read_csv('test_(2)_(1)_(1).csv')

# Clean location names
train_df['location'] = train_df['location'].str.strip().str.lower()
test_df['location'] = test_df['location'].str.strip().str.lower()
dist_df['location'] = dist_df['location'].str.strip().str.lower()
rent_df['location'] = rent_df['location'].str.strip().str.lower()

# Fix test data typos
test_df['location'] = test_df['location'].replace({'kanakapura  rod': 'kanakpura road', 'cooketown': 'cooke town'})

# Group rare locations (less than 10 occurrences in train)
location_counts = train_df['location'].value_counts()
rare_locations = location_counts[location_counts < 10].index
train_df['location'] = train_df['location'].replace(rare_locations, 'other')
test_df['location'] = test_df['location'].apply(lambda x: x if x in location_counts.index and location_counts[x] >= 10 else 'other')

# Merge new features
train_df = train_df.merge(dist_df, on='location', how='left')
train_df = train_df.merge(rent_df, on='location', how='left')
test_df = test_df.merge(dist_df, on='location', how='left')
test_df = test_df.merge(rent_df, on='location', how='left')

# Handle missing values for new features
train_df['dist_from_city'] = train_df['dist_from_city'].fillna(train_df['dist_from_city'].median())
train_df['avg_2bhk_rent'] = train_df['avg_2bhk_rent'].fillna(train_df['avg_2bhk_rent'].median())
test_df['dist_from_city'] = test_df['dist_from_city'].fillna(train_df['dist_from_city'].median())
test_df['avg_2bhk_rent'] = test_df['avg_2bhk_rent'].fillna(train_df['avg_2bhk_rent'].median())

# Cap outliers (94th percentile)
train_df['avg_2bhk_rent'] = train_df['avg_2bhk_rent'].clip(upper=train_df['avg_2bhk_rent'].quantile(0.94))
test_df['avg_2bhk_rent'] = test_df['avg_2bhk_rent'].clip(upper=train_df['avg_2bhk_rent'].quantile(0.94))
train_df['price'] = train_df['price'].clip(upper=train_df['price'].quantile(0.94))

# Clean total_sqft
def clean_total_sqft(sqft):
    try:
        sqft = str(sqft).strip()
        if sqft.replace('.', '').isdigit():
            return float(sqft)
        if '-' in sqft:
            start, end = map(float, sqft.split('-'))
            return (start + end) / 2
        match = re.match(r'(\d+\.?\d*)\s*([A-Za-z\s\.]+)?', sqft)
        if match:
            value = float(match.group(1))
            unit = match.group(2).lower().replace(' ', '') if match.group(2) else ''
            if unit in ['sq.meter', 'sqmeter', 'sq.m', 'sqm']:
                return value * 10.7639
            elif unit in ['sq.yards', 'sqyard', 'sq.y', 'sqy']:
                return value * 9
            elif unit in ['acres', 'acre']:
                return value * 43560
            elif unit in ['perch']:
                return value * 272.25
            elif unit in ['cents', 'cent']:
                return value * 435.6
            elif unit in ['guntha']:
                return value * 1089
            elif unit == '' or unit in ['sq.ft', 'sqft', 'squarefeet']:
                return value
        return np.nan
    except:
        return np.nan

train_df['total_sqft'] = train_df['total_sqft'].apply(clean_total_sqft)
train_df['total_sqft'] = train_df['total_sqft'].clip(upper=train_df['total_sqft'].quantile(0.94))
train_df['total_sqft'] = train_df['total_sqft'].fillna(train_df['total_sqft'].median())
test_df['total_sqft'] = test_df['total_sqft'].apply(clean_total_sqft)
test_df['total_sqft'] = test_df['total_sqft'].clip(upper=train_df['total_sqft'].quantile(0.94))
test_df['total_sqft'] = test_df['total_sqft'].fillna(train_df['total_sqft'].median())

# Standardize size
def standardize_size(size):
    try:
        if pd.isna(size) or not isinstance(size, str) or size.strip() == '':
            return np.nan
        size = size.strip().lower()
        match = re.match(r'(\d+)', size)
        if match:
            number = match.group(1)
            return f"{number} BHK"
        if 'studio' in size:
            return "1 BHK"
        if 'rk' in size:
            return f"{size[0]} BHK"
        return np.nan
    except:
        return np.nan

train_df['size'] = train_df['size'].apply(standardize_size)
train_df['size'] = train_df['size'].fillna(train_df['size'].mode()[0])
test_df['size'] = test_df['size'].apply(standardize_size)
test_df['size'] = test_df['size'].fillna(train_df['size'].mode()[0])

# Handle other missing values
train_df['bath'] = train_df['bath'].fillna(train_df['bath'].median())
train_df['balcony'] = train_df['balcony'].fillna(train_df['balcony'].median())
train_df['society'] = train_df['society'].fillna('Unknown')
train_df['location'] = train_df['location'].fillna('other')
test_df['bath'] = test_df['bath'].fillna(train_df['bath'].median())
test_df['balcony'] = test_df['balcony'].fillna(train_df['balcony'].median())
test_df['society'] = test_df['society'].fillna('Unknown')
test_df['location'] = test_df['location'].fillna('other')

# Simplify availability
train_df['availability'] = train_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')
test_df['availability'] = test_df['availability'].apply(lambda x: 'Ready' if x == 'Ready To Move' else 'Not Ready')

# Feature engineering
train_df['sqft_dist_interaction'] = train_df['total_sqft'] * train_df['dist_from_city']
test_df['sqft_dist_interaction'] = test_df['total_sqft'] * test_df['dist_from_city']

# Drop unnecessary columns
train_df = train_df.drop(['ID', 'society'], axis=1)
test_ids = test_df['ID']
test_df = test_df.drop(['ID', 'society'], axis=1)

# Encode categorical variables
categorical_cols = ['area_type', 'availability', 'location', 'size']
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align test data columns
test_df = test_df.reindex(columns=train_df.columns.drop('price'), fill_value=0)

# Define features and target
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
X_test = test_df

# Log transform target
y_train_log = np.log1p(y_train)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Feature selection using XGBoost
initial_model = XGBRegressor(n_estimators=100, random_state=42)
initial_model.fit(X_train_scaled, y_train_log)
importances = pd.DataFrame({'Feature': X_train.columns, 'Importance': initial_model.feature_importances_})
low_importance = importances[importances['Importance'] < 0.015]['Feature']
X_train = X_train.drop(columns=low_importance)
X_test = X_test.drop(columns=low_importance, errors='ignore')
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
# Stacking model
estimators = [
    ('rf', RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42))
]
model = StackingRegressor(
    estimators=estimators,
    final_estimator=XGBRegressor(n_estimators=50, random_state=42),
    cv=5
)
model.fit(X_train_scaled, y_train_log)

StackingRegressor(cv=5,
                  estimators=[('rf',
                               RandomForestRegressor(max_depth=15,
                                                     n_estimators=200,
                                                     random_state=42)),
                              ('xgb',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None,
                                            feature_...
                                               feature_weights=None, gamma=None,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=None, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None,
                                               max_depth=None, max_leaves=None,
                                               min_child_weight=None,
                                               missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=50, n_jobs=None,
                                               num_parallel_tree=None, ...))

In [5]:
# Predict on test data and reverse log transformation
test_predictions_log = model.predict(X_test_scaled)
test_predictions = np.expm1(np.maximum(test_predictions_log, 0))
predictions_df = pd.DataFrame({'ID': test_ids, 'price': test_predictions})
predictions_df.to_csv('test_predictions_stacking_log.csv', index=False)
print("Predictions saved to 'test_predictions_stacking_log.csv'")

Predictions saved to 'test_predictions_stacking_log.csv'
